# TamilScript AI: TrOCR Fine-Tuning on Real Tamil Handwriting (GPU)

This notebook trains **TrOCR** with a **`microsoft/trocr-base-handwritten`** vision encoder and a dedicated **Tamil RoBERTa decoder (`d42kw01f/Tamil-RoBERTa`)** on the full **`c3rl/IIIT-INDIC-HW-WORDS-Tamil`** dataset (75.7k real handwritten words).

### Requirements:
- Runtime: Google Colab **T4 GPU** (Hardware Accelerator)
- Mixed Precision: **FP16**
- Optimizer: AdamW (`encoder_lr=2e-5`, `decoder_lr=5e-5`), Cosine warmup
- Effective Batch Size: 32 (via Gradient Accumulation)
- **GATE 1 Target**: Validation CER < 15% (Target < 10%)

In [ ]:
# Step 1: Verify GPU & Install Dependencies
!nvidia-smi
!pip install -q transformers torch torchvision datasets pyarrow huggingface_hub evaluate tensorboard

In [ ]:
# Step 2: Download Dataset from Hugging Face Hub (IIIT-INDIC-HW-WORDS-Tamil)
from huggingface_hub import hf_hub_download
import os

repo_id = "c3rl/IIIT-INDIC-HW-WORDS-Tamil"
os.makedirs("data", exist_ok=True)

# Download train & test parquet splits
test_file = hf_hub_download(repo_id=repo_id, filename="data/test-00000-of-00002.parquet", repo_type="dataset", local_dir=".")
train_file = hf_hub_download(repo_id=repo_id, filename="data/train-00000-of-00008.parquet", repo_type="dataset", local_dir=".")
print(f"[*] Downloaded test file: {test_file}")
print(f"[*] Downloaded train file: {train_file}")

In [ ]:
# Step 3: Run Fast Convergence Check (Overfit 200 samples to confirm loss drops)
!python ml/train/train_trocr_tamil.py \
    --encoder microsoft/trocr-base-handwritten \
    --decoder d42kw01f/Tamil-RoBERTa \
    --epochs 5 \
    --batch_size 8 \
    --grad_accum 4 \
    --overfit_200 \
    --fp16

In [ ]:
# Step 4: Full Training on Real Handwriting Dataset (Gate 1: Validation CER < 15%)
!python ml/train/train_trocr_tamil.py \
    --encoder microsoft/trocr-base-handwritten \
    --decoder d42kw01f/Tamil-RoBERTa \
    --epochs 15 \
    --batch_size 16 \
    --grad_accum 2 \
    --encoder_lr 2e-5 \
    --decoder_lr 5e-5 \
    --fp16 \
    --save_dir checkpoints

In [ ]:
# Step 5: Export Fine-Tuned Model & Compute Final Test CER/WER
from transformers import VisionEncoderDecoderModel, TrOCRProcessor, AutoTokenizer

model_path = "checkpoints/trocr_tamil_best"
model = VisionEncoderDecoderModel.from_pretrained(model_path)
processor = TrOCRProcessor.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
print("[*] Best TrOCR checkpoint loaded and ready for deployment.")